# Step 09.4 — GRU：用单一 Hidden State 做门控更新

Step 9.3 已经学过 LSTM：它用 `h_t` 与 `c_t` 两个 recurrent states，加上 forget / input / output gates 来控制保留、写入与暴露。

GRU（Gated Recurrent Unit）的目标不是简单复制 LSTM，而是用**更少的 state 与 gate 结构**实现类似的门控思想。

继续沿用：`B=batch size`、`T=sequence length`、`t=0...T-1`、`C=input feature width`、`H=hidden width`，初态记作 `h_-1`。

本节只准备 Step 9.4；不进入 Step 9.5。


## 1. 从 LSTM 到 GRU：为什么可以只保留一个 Recurrent State？

LSTM 有两个 states：

```text
h_t : hidden state
c_t : cell state
```

GRU 则只维护：

```text
h_t : hidden state
```

也就是说 GRU 没有独立的 `c_t`。它把“保留旧状态”和“写入新 candidate”的控制直接合并到 `h_t` 的更新里。

GRU 主要有两个 sigmoid gates：

```text
r_t : reset gate
z_t : update gate
```

以及一个 tanh candidate：

```text
n_t : candidate hidden state
```

直觉上：

- `r_t` 决定在构造 candidate 时，旧 hidden information 参与多少；
- `z_t` 决定最终 `h_t` 更偏向旧 `h_(t-1)`，还是新 candidate `n_t`。

这使 GRU 成为一种更紧凑的 gated recurrent architecture。它不是“必然优于 LSTM”的版本；不同任务、数据与优化条件下二者表现可能不同。


## 2. GRU 的三个分量：Reset Gate、Update Gate 与 Candidate

本课程采用与 MLX 0.32.2 一致的更新约定。对当前输入 `x_t` 与上一状态 `h_(t-1)`：

$$
r_t=\sigma(W_{xr}x_t+W_{hr}h_{t-1}+b_r)
$$

$$
z_t=\sigma(W_{xz}x_t+W_{hz}h_{t-1}+b_z)
$$

candidate：

$$
n_t=\tanh\left(
W_{xn}x_t+b_n
+
r_t\odot(W_{hn}h_{t-1}+b_{hn})
\right)
$$

最终 hidden update：

$$
\boxed{
h_t
=
(1-z_t)\odot n_t
+
z_t\odot h_{t-1}
}
$$

### Update Gate `z_t` 的含义

因为 `z_t` 来自 sigmoid，所以每个 coordinate 都在 `(0,1)`。

在当前约定下：

```text
z_t ≈ 1 -> 更保留旧 h_(t-1)
z_t ≈ 0 -> 更采用新 candidate n_t
```

这是一个逐 coordinate 的 weighted interpolation（加权插值）：两个系数 `z_t` 与 `1-z_t` 相加为 1。

不同资料有时会交换 update-gate 的命名/公式方向，因此阅读实现时应先看实际方程；本课程后续统一使用上面的 MLX 0.32.2 约定。

### Reset Gate `r_t` 的含义

`r_t` 不直接决定最终保留多少旧 state。它控制的是：**构造 candidate `n_t` 时，旧 hidden projection 参与多少。**

若某个 coordinate `r_t≈0`，旧 hidden contribution 在 candidate path 中被明显抑制；若 `r_t≈1`，candidate 可以较充分利用上一状态。


In [ ]:
import math

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

# Tiny scalar update-gate example
h_prev = 0.8
n_t = 0.2

for z_t in [0.9, 0.1]:
    h_t = (1.0 - z_t) * n_t + z_t * h_prev
    print(f'z={z_t:.1f} -> h_t={h_t:.3f}')

# Tiny reset-gate intuition: same input term and recurrent proposal
input_term = 0.1
recurrent_proposal = 0.6

for r_t in [0.2, 0.9]:
    candidate_pre = input_term + r_t * recurrent_proposal
    candidate = math.tanh(candidate_pre)
    print(
        f'r={r_t:.1f} -> candidate_pre={candidate_pre:.3f}, '
        f'n={candidate:.3f}'
    )


## 3. Shape 与 `3H` Packed Projection：为什么 GRU 比 LSTM 少一组？

输入与 state：

```text
x_t       : (B,C)
h_(t-1)   : (B,H)
```

`r_t`、`z_t`、`n_t` 都是：

```text
(B,H)
```

因此工程实现常把三组 input projections 打包为：

$$
W_x\in\mathbb R^{3H\times C}
$$

recurrent projections 打包为：

$$
W_h\in\mathbb R^{3H\times H}.
$$

MLX row/batch convention：

```text
x_t @ W_x.T       -> (B,3H)
h_prev @ W_h.T    -> (B,3H)
split             -> r / z / n branches, each (B,H)
```

这里的 `3H` 只是把 reset、update、candidate 三组 H-dimensional projection 打包在一起。

对比 LSTM：

```text
LSTM -> i / f / g / o -> 4H
GRU  -> r / z / n     -> 3H
```

GRU 也只维护一个 `h_t`，不再维护单独的 `c_t`。

### 一个透明的教学版 GRU Cell

下面的 `EducationalGRUCell` 把 input projection 打包成 `3H`，同时把 recurrent `r/z` 与 candidate branch 分开写，方便看到 reset gate 只作用在 candidate recurrent contribution 上。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

class EducationalGRUCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size

        # Input branches: r, z, n -> 3H
        self.x_proj = nn.Linear(input_size, 3 * hidden_size, bias=True)

        # Recurrent branches for r and z -> 2H
        self.h_rz = nn.Linear(hidden_size, 2 * hidden_size, bias=False)

        # Recurrent candidate branch -> H, with its own bias
        self.h_n = nn.Linear(hidden_size, hidden_size, bias=True)

    def __call__(self, x_t, h_prev):
        x_r, x_z, x_n = mx.split(self.x_proj(x_t), 3, axis=-1)
        h_r, h_z = mx.split(self.h_rz(h_prev), 2, axis=-1)

        r_t = mx.sigmoid(x_r + h_r)
        z_t = mx.sigmoid(x_z + h_z)

        n_t = mx.tanh(x_n + r_t * self.h_n(h_prev))
        h_t = (1.0 - z_t) * n_t + z_t * h_prev

        return h_t, (r_t, z_t, n_t)

B, C, H = 2, 3, 4
cell = EducationalGRUCell(C, H)
x_t = mx.zeros((B, C))
h_prev = mx.zeros((B, H))
h_t, parts = cell(x_t, h_prev)

print('x_t:', x_t.shape)
print('h_prev / h_t:', h_prev.shape, h_t.shape)
print('r / z / n shapes:', [p.shape for p in parts])


## 4. Update Gate 与长期 Gradient：它也提供一条可学习的 Direct Retain Path

GRU update：

$$
h_t=(1-z_t)\odot n_t+z_t\odot h_{t-1}.
$$

只看计算图中最直接的：

```text
h_(t-1) -- multiply by z_t --> h_t
```

并在这个局部 partial derivative 中把当前 `z_t` 与 `n_t` 当作给定的其他输入，则 direct Jacobian contribution 是：

$$
\boxed{
\left.
\frac{\partial h_t}{\partial h_{t-1}}
\right|_{direct}
=
\operatorname{diag}(z_t)
}
$$

所以 direct backward contribution：

$$
g_{h_{t-1}}^{direct}
=
z_t\odot g_{h_t}.
$$

如果某个 coordinate `z_t≈1`，旧 hidden state 与这条 direct gradient contribution 都更容易被保留；如果 `z_t≈0`，最终 state 更偏向 candidate。

这与 LSTM 的 forget-gate direct cell path 很相似，但 GRU 把控制放在同一个 hidden state 上。

同样要保留严谨边界：

- `diag(z_t)` 只是 direct retain edge contribution；
- `z_t`、`r_t`、`n_t` 都依赖 `h_(t-1)`，完整 GRU Jacobian 还有其他路径；
- `z_t≈1` 能提供接近 identity 的 direct path，但不保证整个网络永不出现 vanishing/exploding；
- reset gate 主要影响 candidate 构造，不等于“直接删除 hidden state”。


In [ ]:
z_t = mx.array([0.95, 0.20])
g_h_t = mx.array([2.0, -3.0])

g_h_prev_direct = z_t * g_h_t

print('update gate:', z_t)
print('upstream hidden gradient:', g_h_t)
print('direct gradient contribution:', g_h_prev_direct)


## 5. MLX 0.32.2 `nn.GRU` 与 LSTM 的结构/参数对比

项目固定 `mlx==0.32.2`。这一版本提供：

```python
nn.GRU(input_size, hidden_size, bias=True)
```

输入 sequence 可以是 `(B,T,C)`；可选 initial `hidden` 为 `(B,H)`。返回：

```text
all_hidden : (B,T,H)
```

没有额外的 `all_cell`，因为 GRU 没有独立 cell state。

MLX 0.32.2 源码中的 packed parameters：

```text
Wx  : (3H,C)
Wh  : (3H,H)
b   : (3H,)   # bias=True
bhn : (H,)    # candidate recurrent branch bias
```

源码按 `r / z` 与 `n` 分支处理，其中 candidate 使用：

$$
n_t=\tanh(x_n+r_t\odot h_n).
$$

### 与 LSTM 比参数量

在相同 C、H 且 `bias=True` 的 MLX 0.32.2 实现中：

$$
P_{GRU}=3HC+3H^2+4H
$$

$$
P_{LSTM}=4HC+4H^2+4H.
$$

例如 `C=3, H=4`：

```text
GRU  = 100 parameters
LSTM = 128 parameters
```

所以在相同 input/hidden width 下，GRU 的主要 weight matrices 更少。

但“参数更少”不等于“效果一定更好”或“训练一定更快”；真实表现仍受任务、序列长度、实现、硬件与优化过程影响。


In [ ]:
B, T, C, H = 2, 5, 3, 4

gru = nn.GRU(
    input_size=C,
    hidden_size=H,
    bias=True,
)

X = mx.zeros((B, T, C))
h_minus_1 = mx.zeros((B, H))

all_hidden = gru(X, hidden=h_minus_1)

gru_parameter_count = (
    gru.Wx.size + gru.Wh.size + gru.b.size + gru.bhn.size
)

print('X:', X.shape)
print('Wx / Wh / b / bhn:',
      gru.Wx.shape, gru.Wh.shape, gru.b.shape, gru.bhn.shape)
print('all_hidden:', all_hidden.shape)
print('GRU parameter count:', gru_parameter_count)


### Step 9.4 材料边界

- GRU 只有一个 recurrent hidden state `h_t`，不维护独立 `c_t`。
- `r_t` 是 reset gate，主要控制旧 hidden contribution 如何参与 candidate 构造。
- `z_t` 是 update gate；按本课程/MLX 0.32.2 约定，`z_t≈1` 更保留旧 state，`z_t≈0` 更采用新 candidate。
- candidate `n_t` 使用 tanh；最终更新是 `(1-z_t)⊙n_t + z_t⊙h_(t-1)`。
- direct retain edge 的 Jacobian contribution 是 `diag(z_t)`，但它不是完整 GRU total Jacobian。
- packed projection 从 LSTM 的 `4H` 变为 GRU 的 `3H`；相同 C/H 下 GRU 通常有更少的主要 recurrent/input weights。
- MLX 0.32.2 `nn.GRU` 返回 hidden-state sequence `(B,T,H)`，没有 cell-state sequence。
- 本轮只准备 Step 9.4；不进入 Step 9.5，也不据此声明学员已经掌握或完成练习。
